# UCF-101 Verification for RGB 32 x 32 x 30
Best model epoch x, Model clean accuracy x%

# Temporal Systems with shared ε-constraints and Lipschitz bound propagation
The following models and experiments propose verifying with specific ε constraints. This constraints produce lesser perturbations than applying the same ε constraints on the entire temporal system. However, they allow the model to be verifiable, make the model more robust and reduce verification error.

The variants of shared ε constraints we aim to propose are as follows:
- Shared ε constraints on fixed / contiguous 'k' frames
- Shared ε constraints on arbitrary 'k' frames
- Shared ε constraints on 'c' sets of 'k' fixed / contiguuous frames
- Shared ε constraints on 'c' sets of 'k' arbitrary frames

We hope to improve the robustness of the model using these techniques and evaluate robustness of the variations across several temporal systems.



### UCF-101 Action Recognition
[UCF-101](https://www.crcv.ucf.edu/data/UCF101.php) is a action recognition dataset with videos categorized into 101 action classes.

There are 5 types of action categories, namely:
- Human-Object Interaction
- Body-Motion Only
- Human-Human Interaction
- Playing Musical Instruments
- Sports

This dataset contains 13320 videos, each video with a frame rate of 25 fps and a resolution of 320 x 240.


To achieve this task, we sample the videos to obtain frames. Videos have different lengths and the shortest one has 28 frames. In order to reduce the preprocessing overhead of converting the videos into frames (using [OpenCV](https://opencv.org/) or [FFmpeg](https://www.ffmpeg.org/)), we use the preprocessed dataset from [feichtenhofer](https://github.com/feichtenhofer/twostreamfusion).

Specifically, we use the dataset for RGB **UCF101 RGB:** 
[**part1**](http://ftp.tugraz.at/pub/feichtenhofer/tsfusion/data/ucf101_jpegs_256.zip.001),
[**part2**](http://ftp.tugraz.at/pub/feichtenhofer/tsfusion/data/ucf101_jpegs_256.zip.002),
[**part3**](http://ftp.tugraz.at/pub/feichtenhofer/tsfusion/data/ucf101_jpegs_256.zip.003)
Extract and unzip the data to `ucf101_jpegs_256`

The model that performs action classification uses a 3DCNN Model that is adapted from [HHTseng](https://github.com/HHTseng/video-classification/tree/master/Conv3D). The model is available in the `models` folder.

In [30]:
%load_ext autotime 
#to measure cell execution time of all cells

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 993 ms (started: 2026-03-30 12:47:33 +01:00)


In [31]:

import os
import sys
from pathlib import Path

PWD = Path(os.path.abspath('')).parent.parent.resolve()

# DATA_PATH = Path.home() / "data/ucf101_jpegs_256/" # Change the data path as needed
DATA_PATH = "/vol/bitbucket/svadakku/data/ucf101_jpegs_256/" # Change the data path as needed
BOUNDING_BOX_PATH = "/vol/bitbucket/svadakku/data/ucf101_jpegs_256_tracking/"  # load preprocessed bounding boxes
BOUNDING_BOX_JSON_FILE_NAME = "all_objects_bounding_box_tracked.json"  # load preprocessed bounding boxes

MODELS_PATH = Path(os.path.abspath('')).parent.parent.resolve() / "models/"

SAVED_MODELS_PATH = MODELS_PATH / "3dcnn_1frame_skip_32_32_rgb_5_class"
Path(SAVED_MODELS_PATH).mkdir(exist_ok=True, parents=True)

ACTION_PATH = MODELS_PATH / "UCF101actions.pkl"  # load preprocessed action names
ACTION_CATEGORY_PATH = MODELS_PATH / "UCF101ActionCategories.pkl"  # load preprocessed action categories


# Include python scripts in the parent folder
module_path = os.path.abspath(os.path.abspath('../..'))    

if module_path not in sys.path:       
    sys.path.append(module_path)

# Auto reload python modules, useful for editing the modules with a running kernel
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
time: 89.5 ms (started: 2026-03-30 12:47:34 +01:00)


In [32]:
import torch
import numpy as np

# Detect devices
use_cuda = torch.cuda.is_available()                   # check if GPU exists
device = torch.device("cuda:1" if use_cuda else "cpu")   # use CPU or GPU
device

device(type='cuda', index=1)

time: 79.3 ms (started: 2026-03-30 12:47:34 +01:00)


In [33]:

# training parameters
num_classes = 5   # Reduce the number of classes to 5 Action Categories
epochs = 50
batch_size = 1000
learning_rate = 1e-3
log_interval = 10
# img_x, img_y = 256, 342  # resize video 2d frame size (original: 256 x 342)

img_x, img_y = 32, 32 
num_channels = 3
num_frames = 5

# 3D CNN parameters
fc_hidden1, fc_hidden2 = 32, 32
dropout = 0.5       # dropout probability

# Select which frame to begin & end in videos
begin_frame, end_frame, skip_frame = 1, 29, 1

selected_frames = np.arange(begin_frame, end_frame, skip_frame).tolist()

time: 63 ms (started: 2026-03-30 12:47:34 +01:00)


In [34]:
import sys
import os

import pickle
from sklearn.model_selection import train_test_split
from utils.data_utils_3dcnn import *
import torchvision.transforms as transforms
import torch.utils.data as data
import numpy as np


time: 67.1 ms (started: 2026-03-30 12:47:34 +01:00)


## Training the 3D CNN - MODEL
The trained model is present in `models` folder. 
Skip to the Verification section if re-training is not needed.

In [35]:
(all_X_list, all_y_list) = load_file_names_and_category_labels(DATA_PATH, ACTION_PATH, ACTION_CATEGORY_PATH)

# train, test split
train_list, test_list, train_label, test_label = train_test_split(all_X_list, all_y_list, test_size=0.25, random_state=42)

# save the test, train splits for reproducibility
with open(SAVED_MODELS_PATH / "train_indices.pkl", 'wb') as handle:
    pickle.dump(train_list, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(SAVED_MODELS_PATH / "test_indices.pkl", 'wb') as handle:
    pickle.dump(test_list, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(SAVED_MODELS_PATH / "train_labels.pkl", 'wb') as handle:
    pickle.dump(train_label, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(SAVED_MODELS_PATH / "test_labels.pkl", 'wb') as handle:
    pickle.dump(test_label, handle, protocol=pickle.HIGHEST_PROTOCOL)
    

# image transformation
transform = transforms.Compose([transforms.Resize([img_x, img_y]),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.5], std=[0.5])])

# load UCF101 actions names
params = {'batch_size': batch_size, 'shuffle': False, 'num_workers': 4, 'pin_memory': True} if use_cuda else {}

# Set selected_frames as None so that all the frames are used - results in non-uniform tensor sizes
train_set, valid_set = Dataset_3DCNN_MidFrames(DATA_PATH, train_list, train_label, None, transform=transform, is_image_names_needed=False, num_frames=num_frames, is_RGB=True), \
                       Dataset_3DCNN_MidFrames(DATA_PATH, test_list, test_label, None, transform=transform, is_image_names_needed=False, num_frames=num_frames, is_RGB=True)


train_loader = data.DataLoader(train_set, **params)
valid_loader = data.DataLoader(valid_set, **params)

time: 92.5 ms (started: 2026-03-30 12:47:34 +01:00)


In [36]:
# Load the exisiting test / train split on the dataset
with open(SAVED_MODELS_PATH / "test_indices.pkl", 'rb') as handle:
    test_list = pickle.load(handle)


with open(SAVED_MODELS_PATH / "test_labels.pkl", 'rb') as handle:
    test_label = pickle.load(handle)

with open(SAVED_MODELS_PATH / "train_indices.pkl", 'rb') as handle:
    train_list = pickle.load(handle)


with open(SAVED_MODELS_PATH / "train_labels.pkl", 'rb') as handle:
    train_label = pickle.load(handle)
    
# image transformation
transform = transforms.Compose([transforms.Resize([img_x, img_y]),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.5], std=[0.5])])

# Set selected_frames as None so that all the frames are used - results in non-uniform tensor sizes
train_set, valid_set = Dataset_3DCNN_MidFrames(DATA_PATH, train_list, train_label, None, transform=transform, is_image_names_needed=False, num_frames=num_frames, is_RGB=True), \
                       Dataset_3DCNN_MidFrames(DATA_PATH, test_list, test_label, None, transform=transform, is_image_names_needed=False, num_frames=num_frames, is_RGB=True)

# load UCF101 actions names
params = {'batch_size': batch_size, 'shuffle': False, 'num_workers': 4, 'pin_memory': True} if use_cuda else {}
                       
train_loader = data.DataLoader(train_set, **params)
valid_loader = data.DataLoader(valid_set, **params)

val_loader = valid_loader
DEVICE = device

time: 67.6 ms (started: 2026-03-30 12:47:34 +01:00)


In [37]:

'''
# %%capture cap --no-stderr 
# # Capture the output and write to file

from models.models import *
from utils.train_validate_3dcnn import *

# create model
cnn3d = CNN3D(t_dim=len(selected_frames), img_x=img_x, img_y=img_y,
              drop_p=dropout, fc_hidden1=fc_hidden1,  fc_hidden2=fc_hidden2, 
              num_classes=num_classes, in_channels=num_channels,
              ).to(device)


# Parallelize model to multiple GPUs
# if torch.cuda.device_count() > 1:
#     print("Using", torch.cuda.device_count(), "GPUs!")
#     cnn3d = nn.DataParallel(cnn3d)

optimizer = torch.optim.Adam(cnn3d.parameters(), lr=learning_rate)   # optimize all cnn parameters
criterion = nn.CrossEntropyLoss()
'''

'\n# %%capture cap --no-stderr \n# # Capture the output and write to file\n\nfrom models.models import *\nfrom utils.train_validate_3dcnn import *\n\n# create model\ncnn3d = CNN3D(t_dim=len(selected_frames), img_x=img_x, img_y=img_y,\n              drop_p=dropout, fc_hidden1=fc_hidden1,  fc_hidden2=fc_hidden2, \n              num_classes=num_classes, in_channels=num_channels,\n              ).to(device)\n\n\n# Parallelize model to multiple GPUs\n# if torch.cuda.device_count() > 1:\n#     print("Using", torch.cuda.device_count(), "GPUs!")\n#     cnn3d = nn.DataParallel(cnn3d)\n\noptimizer = torch.optim.Adam(cnn3d.parameters(), lr=learning_rate)   # optimize all cnn parameters\ncriterion = nn.CrossEntropyLoss()\n'

time: 61.5 ms (started: 2026-03-30 12:47:34 +01:00)


In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional

class UCF101SimpleCNN(nn.Module):
    """
    A simple 3D CNN model for UCF-101 action recognition.
    
    Architecture:
    - Only 3 convolutional layers
    - Global average pooling to reduce parameters
    - Single fully connected layer for classification
    """
    def __init__(
        self, 
        num_frames: int = 16,
        in_channels= 3,
        img_x=342, 
        img_y=256,
        num_classes: int = 101,
        dropout_rate: float = 0.5
    ):
        super(UCF101SimpleCNN, self).__init__()
        
        # Model configuration
        self.num_frames = num_frames
        self.input_shape = (img_x, img_y)
        self.in_channels = in_channels
        self.img_x = img_x
        self.img_y = img_y
        self.dropout_rate = dropout_rate
        self.num_classes = num_classes
        self.name = "ucf101_simple_cnn"
        
        # Define the convolutional layers - keep it simple with just 3 layers
        # First conv layer: reduce spatial dimensions with stride 2
        self.conv1 = nn.Conv3d(self.in_channels, 16, kernel_size=(3, 5, 5), stride=(1, 2, 2), padding=(0, 0, 0))
        
        self.layer1 = nn.Sequential(
            self.conv1,
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )
        
        # Second conv layer
        self.conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        
        self.layer2 = nn.Sequential(
            self.conv2,
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )
        
        # Third conv layer
        self.conv3 = nn.Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        self.layer3 = nn.Sequential(
            self.conv3,
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)) 
        )
        
        # Global average pooling
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes)
        )
        
        # Initialize weights
        self._initialize_weights()
    
    def forward(self, x):
        # Input shape: (batch_size, channels, frames, height, width)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        # Global pooling
        x = self.global_pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Classification
        x = self.classifier(x)
        
        return x
    
    def _initialize_weights(self):
        """Initialize model weights for better convergence"""
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    @classmethod
    def from_pretrained(cls, checkpoint_path, **kwargs):
        """Load a pretrained model from checkpoint"""
        model = cls(**kwargs)
        model.load_state_dict(torch.load(checkpoint_path))
        return model

time: 62.8 ms (started: 2026-03-30 12:47:34 +01:00)


In [39]:
# create model
cnn3d = UCF101SimpleCNN(num_frames=num_frames, img_x=img_x, img_y=img_y,
              num_classes=num_classes, in_channels=num_channels, dropout_rate=dropout,
              ).to(device)



# Parallelize model to multiple GPUs
# if torch.cuda.device_count() > 1:
#     print("Using", torch.cuda.device_count(), "GPUs!")
#     cnn3d = nn.DataParallel(cnn3d)

optimizer = torch.optim.Adam(cnn3d.parameters(), lr=learning_rate)   # optimize all cnn parameters
criterion = nn.CrossEntropyLoss()

time: 62.2 ms (started: 2026-03-30 12:47:34 +01:00)


In [28]:
from torch.utils.data import DataLoader
from tqdm import tqdm
from typing import Tuple
from torch import nn

def evaluate(model: nn.Module, data_loader: DataLoader, criterion: nn.Module, device: torch.device) -> Tuple[float, float]:
    """Evaluate the model on the given data loader"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            target = target.squeeze()
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    
    acc = 100. * correct / total
    avg_loss = total_loss / len(data_loader)
    return acc, avg_loss

def train(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, 
          optimizer: torch.optim.Optimizer, criterion: nn.Module, 
          device: torch.device, model_path: str, epochs: int = 10) -> None:
    """Train the model and save the best checkpoint based on validation accuracy
    
    Args:
        model: The model to train
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        optimizer: Optimizer to use for training
        criterion: Loss function
        device: Device to train on
        model_path: Path where the best model checkpoint will be saved
        epochs: Number of epochs to train for
    """
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(train_loader)):
            data, target = data.to(device), target.to(device)
            target = target.squeeze()
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
        
        train_acc = 100. * correct / total
        train_loss = total_loss / len(train_loader)
        
        # Validation phase
        val_acc, val_loss = evaluate(model, val_loader, criterion, device)
        
        print(f'Epoch {epoch}:')
        print(f'  Training: Loss = {train_loss:.4f}, Accuracy = {train_acc:.2f}%')
        print(f'  Validation: Loss = {val_loss:.4f}, Accuracy = {val_acc:.2f}%')
        
        # Save model if this is the best validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            print(f'New best validation accuracy: {best_val_acc:.2f}%! Saving model...')
            torch.save(model.state_dict(), model_path)
    
    print(f'\nTraining complete! Best validation accuracy: {best_val_acc:.2f}%')
    print(f'Best model saved to {model_path}') 

time: 55.3 ms (started: 2026-03-30 12:21:31 +01:00)


In [29]:
%%capture cap --no-stderr 
# Capture the output and write to file
model_path = SAVED_MODELS_PATH / "best_simple3dcnn_ucf101.pth"
train(cnn3d, train_loader, valid_loader, optimizer, criterion, device, model_path=model_path, epochs=epochs)

  0%|          | 0/10 [00:00<?, ?it/s]

 40%|████      | 4/10 [00:08<00:13,  2.22s/it]/vol/bitbucket/svadakku/miniconda3/envs/bit-vas/lib/python3.11/site-packages/PIL/Image.py:3465: ResourceWarning: unclosed file <_io.BufferedReader name='/vol/bitbucket/svadakku/data/ucf101_jpegs_256/v_PlayingViolin_g18_c01/frame000084.jpg'>
  fp = builtins.open(filename, "rb")
/vol/bitbucket/svadakku/miniconda3/envs/bit-vas/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py:326: ResourceWarning: unclosed file <_io.BufferedReader name='/vol/bitbucket/svadakku/data/ucf101_jpegs_256/v_HammerThrow_g23_c01/frame000104.jpg'>
  pass



KeyboardInterrupt: 

time: 25min 21s (started: 2026-03-30 12:21:31 +01:00)


In [ ]:
with open(SAVED_MODELS_PATH / 'epoc_logs.txt', 'w') as f:
    f.write(cap.stdout)

In [ ]:
with open(SAVED_MODELS_PATH / 'epoc_logs.txt', 'r') as f:
    file_contents = f.read()
    print(file_contents)

Epoch 0:
  Training: Loss = 1.4321, Accuracy = 46.44%
  Validation: Loss = 1.4430, Accuracy = 50.51%
New best validation accuracy: 50.51%! Saving model...
Epoch 1:
  Training: Loss = 1.2313, Accuracy = 52.18%
  Validation: Loss = 1.2758, Accuracy = 52.10%
New best validation accuracy: 52.10%! Saving model...
Epoch 2:
  Training: Loss = 1.1538, Accuracy = 55.13%
  Validation: Loss = 1.1191, Accuracy = 56.73%
New best validation accuracy: 56.73%! Saving model...
Epoch 3:
  Training: Loss = 1.1165, Accuracy = 55.99%
  Validation: Loss = 1.1068, Accuracy = 55.68%
Epoch 4:
  Training: Loss = 1.0898, Accuracy = 57.19%
  Validation: Loss = 1.0647, Accuracy = 58.65%
New best validation accuracy: 58.65%! Saving model...
Epoch 5:
  Training: Loss = 1.0669, Accuracy = 58.81%
  Validation: Loss = 1.0458, Accuracy = 60.24%
New best validation accuracy: 60.24%! Saving model...
Epoch 6:
  Training: Loss = 1.0459, Accuracy = 59.92%
  Validation: Loss = 1.0249, Accuracy = 61.50%
New best validation acc

In [40]:
import random

batch_size = 25

# Set random seed for reproducibility
random.seed(42)

# Select 100 random samples
samples_size = 10
random_test_list = random.sample(test_list, k=samples_size)
random_test_label = [test_label[test_list.index(x)] for x in random_test_list]

valid_set = Dataset_3DCNN_MidFrames(DATA_PATH, test_list, test_label, selected_frames, transform=transform, is_image_names_needed=False, num_frames=num_frames, is_RGB=True)

# load UCF101 actions names
params = {'batch_size': batch_size, 'shuffle': False, 'num_workers': 4, 'pin_memory': True} if use_cuda else {}
                       
valid_loader = data.DataLoader(valid_set, **params)


time: 75.2 ms (started: 2026-03-30 12:47:41 +01:00)


In [41]:
from models.models import *
from utils.train_validate_3dcnn import *
from typing import *

model_path = SAVED_MODELS_PATH / "best_simple3dcnn_ucf101.pth"
# model = CNN3D(t_dim=len(selected_frames), img_x=img_x, img_y=img_y,
#               drop_p=dropout, fc_hidden1=fc_hidden1,  fc_hidden2=fc_hidden2, 
#               num_classes=num_classes, in_channels=num_channels,
#               ).to(device)

model = UCF101SimpleCNN(num_frames=num_frames, img_x=img_x, img_y=img_y,
              num_classes=num_classes, in_channels=num_channels, dropout_rate=dropout,
              ).to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

UCF101SimpleCNN(
  (conv1): Conv3d(3, 16, kernel_size=(3, 5, 5), stride=(1, 2, 2))
  (layer1): Sequential(
    (0): Conv3d(3, 16, kernel_size=(3, 5, 5), stride=(1, 2, 2))
    (1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (layer2): Sequential(
    (0): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (layer3): Sequential(
    (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))

time: 66.8 ms (started: 2026-03-30 12:47:42 +01:00)


## CROWN-IBP

In [42]:

import sys
from copy import deepcopy
from pathlib import Path
# Add path to find the custom operator
project_root = Path('/data/sv/temporal-nn-verification/')
sys.path.append(str(project_root / 'ucf101_classification'))
sys.path.append(str(project_root / 'ucf101_classification/utils'))

from auto_LiRPA import BoundedModule, BoundedTensor, register_custom_op
from auto_LiRPA.perturbations import PerturbationLpNorm
from auto_LiRPA_custom_operators.convolution_3d import BoundConv3D
from auto_LiRPA_custom_operators.pooling_3d import BoundAveragePoolGeneral,BoundMaxPoolGeneral, BoundGlobalAveragePool

# Register custom operator for 3D convolutions to avoid Conv2D type errors
register_custom_op("onnx::Conv", BoundConv3D)
register_custom_op("onnx::MaxPool", BoundMaxPoolGeneral)
register_custom_op("onnx::AveragePool", BoundAveragePoolGeneral)
register_custom_op("onnx::GlobalAveragePool", BoundGlobalAveragePool)
register_custom_op("onnx::AdaptiveAvgPool3d", BoundGlobalAveragePool)
# custom_operators = {
#     'AdaptiveAvgPool3d': BoundAdaptiveAvgPool3d
# }

crown_rob_acc = []
crown_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        model.eval()
        lirpa_model = BoundedModule(
            deepcopy(model),
            torch.empty_like(X),
            # custom_ops = custom_operators,
            device=X.device,
            verbose=False,
        )
        lirpa_model.eval()
        # print(lirpa_model.training)
        
        y = y.to(DEVICE).to(torch.long)
        ptb = PerturbationLpNorm(norm=torch.inf, eps=eps)
        ptb_inputs = BoundedTensor(X, ptb)

        final_lower, final_upper = lirpa_model.compute_bounds(x=(ptb_inputs,), method='CROWN-IBP')
        output = cnn3d(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    
    crown_acc.append((correct / total) * 100)
    crown_rob_acc.append((robust / total) * 100)

print("Model Accuracy: ", crown_acc)
print("CROWN-IBP Robust Accuracy: ", crown_rob_acc)


Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!


/vol/bitbucket/svadakku/miniconda3/envs/bit-vas/lib/python3.11/site-packages/torch/nn/functional.py:2474: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if size_prods == 1:
/vol/bitbucket/svadakku/miniconda3/envs/bit-vas/lib/python3.11/site-packages/torch/onnx/utils.py:631: UserWarning: ONNX Preprocess - Removing mutation from node aten::add_ on block input: '7'. This changes graph semantics. (Triggered internally at ../torch/csrc/jit/passes/onnx/remove_inplace_ops_for_onnx.cpp:350.)
  _C._jit_pass_onnx_remove_inplace_ops_for_onnx(graph, module)
/vol/bitbucket/svadakku/miniconda3/envs/bit-vas/lib/python3.11/site-packages/torch/onnx/utils.py:631: UserWarning: ONNX Preprocess - Removing mutation from node aten::add_ on block input: '14'. This changes graph semantics. (Tri

Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
Set model to train() mode!
Dropout operation CANNOT be parsed during conversion when the model is in eval() mode!
S

### IBP Verification

In [43]:
def propagate_bounds(input_bounds: Tuple[torch.Tensor, torch.Tensor], layer: nn.Module
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Propagate bounds through a single layer using IBP.

        Args:
            input_bounds: Tuple of (lower_bounds, upper_bounds) for the input
            layer: The layer to propagate bounds through

        Returns:
            Tuple of (lower_bounds, upper_bounds) for the output
        """
        lower_bounds, upper_bounds = input_bounds

        if isinstance(layer, nn.Linear):
            return _propagate_linear(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.Conv3d):
            return _propagate_conv3d(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.ReLU):
            return _propagate_relu(lower_bounds, upper_bounds)
        elif isinstance(layer, nn.BatchNorm3d) or isinstance(layer, nn.BatchNorm2d):
            return _propagate_batchnorm(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.Dropout):
            return _propagate_dropout(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.AdaptiveAvgPool3d):
            return _propagate_adaptive_avgpool3d(
                lower_bounds, upper_bounds, layer.output_size
            )
        else:
            raise NotImplementedError(
                f"IBP propagation not implemented for {type(layer)}"
            )

# Convolutional layer IBP
def _propagate_conv3d(lower_bound, upper_bound, conv_layer):
    # Compute center and radius
    x_mu = (lower_bound + upper_bound) / 2
    x_r = (upper_bound - lower_bound) / 2

    # Compute bounds
    W = conv_layer.weight
    abs_W = W.abs()

    conv0 = F.conv3d(
        x_mu,
        W,
        bias=conv_layer.bias,
        stride=conv_layer.stride,
        padding=conv_layer.padding,
        dilation=conv_layer.dilation,
        groups=conv_layer.groups,
    )
    conv1 = F.conv3d(
        x_r,
        abs_W,
        bias=None,
        stride=conv_layer.stride,
        padding=conv_layer.padding,
        dilation=conv_layer.dilation,
        groups=conv_layer.groups,
    )

    z_L = conv0 - conv1
    z_U = conv0 + conv1

    return z_L, z_U

def _propagate_linear(lower_bound, upper_bound, fc_layer):
    # Compute center and radius
    x_mu = (lower_bound + upper_bound) / 2
    x_r = (upper_bound - lower_bound) / 2

    # Compute bounds
    W = fc_layer.weight
    abs_W = W.abs()
    z_L = F.linear(x_mu, W, bias=fc_layer.bias) - F.linear(x_r, abs_W, bias=None)
    z_U = F.linear(x_mu, W, bias=fc_layer.bias) + F.linear(x_r, abs_W, bias=None)

    return z_L, z_U

def _propagate_relu(lower_bound, upper_bound):
    return F.relu(lower_bound), F.relu(upper_bound)

def _propagate_batchnorm(lower_bound, upper_bound, bn_layer):
    running_mean = bn_layer.running_mean
    running_var = bn_layer.running_var

    W = bn_layer.state_dict()["weight"]
    b = bn_layer.state_dict()["bias"]

    # Compute center and radius
    x_mu = (lower_bound + upper_bound) / 2
    x_r = (upper_bound - lower_bound) / 2

    lower_offset = F.batch_norm(
        -x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )
    upper_offset = F.batch_norm(
        x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )

    lower_offset = torch.minimum(lower_offset, upper_offset)
    upper_offset = torch.maximum(lower_offset, upper_offset)

    nominal = F.batch_norm(x_mu, running_mean, running_var, W, b)

    y_l = nominal + lower_offset
    y_u = nominal + upper_offset
    return y_l, y_u

def _propagate_adaptive_avgpool3d(lower_bound, upper_bound, output_size):
    return F.adaptive_avg_pool3d(lower_bound, output_size), F.adaptive_avg_pool3d(
        upper_bound, output_size
    )

def _propagate_dropout(lower_bound, upper_bound, dropout_layer):
    """
    Propagate bounds through a dropout layer.
    
    During verification, we assume the model is in evaluation mode,
    so dropout should effectively be an identity function.
    
    Args:
        lower_bound: Lower bounds of the input
        upper_bound: Upper bounds of the input
        dropout_layer: The dropout layer to propagate through
        
    Returns:
        Tuple of (lower_bounds, upper_bounds) for the output
    """
    if model.training:
        # Only apply scaling during training
        # During training, dropout scales values by 1/(1-p)
        scale = 1.0 / (1.0 - dropout_layer.p)
        return lower_bound * scale, upper_bound * scale
    else:
        # During evaluation, dropout is an identity function
        return lower_bound, upper_bound


def _propagate_conv3d(
        lower_bounds: torch.Tensor, upper_bounds: torch.Tensor, layer: nn.Conv3d
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Propagate bounds through a 3D convolutional layer"""
        # Compute center and radius
        x_mu = (lower_bounds + upper_bounds) / 2
        x_r = (upper_bounds - lower_bounds) / 2

        # Compute bounds
        W = layer.weight
        abs_W = W.abs()

        conv0 = F.conv3d(
            x_mu,
            W,
            bias=layer.bias,
            stride=layer.stride,
            padding=layer.padding,
            dilation=layer.dilation,
            groups=layer.groups,
        )
        conv1 = F.conv3d(
            x_r,
            abs_W,
            bias=None,
            stride=layer.stride,
            padding=layer.padding,
            dilation=layer.dilation,
            groups=layer.groups,
        )

        z_L = conv0 - conv1
        z_U = conv0 + conv1

        return z_L, z_U

def _propagate_batchnorm3d(
    lower_bounds: torch.Tensor,
    upper_bounds: torch.Tensor,
    layer: nn.BatchNorm3d,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Propagate bounds through a 3D batch normalization layer"""
    running_mean = layer.running_mean
    running_var = layer.running_var

    W = layer.state_dict()["weight"]
    b = layer.state_dict()["bias"]

    # Compute center and radius
    x_mu = (lower_bounds + upper_bounds) / 2
    x_r = (upper_bounds - lower_bounds) / 2

    lower_offset = F.batch_norm(
        -x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )
    upper_offset = F.batch_norm(
        x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )

    lower_offset = torch.minimum(lower_offset, upper_offset)
    upper_offset = torch.maximum(lower_offset, upper_offset)

    nominal = F.batch_norm(x_mu, running_mean, running_var, W, b)

    y_l = nominal + lower_offset
    y_u = nominal + upper_offset
    return y_l, y_u

def _propagate_adaptive_avg_pool3d(
    lower_bounds: torch.Tensor,
    upper_bounds: torch.Tensor,
    output_size: Tuple[int, int, int],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Propagate bounds through an adaptive average pooling layer"""
    return (
        F.adaptive_avg_pool3d(lower_bounds, output_size),
        F.adaptive_avg_pool3d(upper_bounds, output_size),
    )
    
def _propagate_maxpool3d(lower_bound, upper_bound, maxpool_layer):
    """
    Propagate bounds through a MaxPool3d layer.
    
    MaxPool is a monotonic function, so we can directly apply it to both bounds.
    
    Args:
        lower_bound: Lower bounds of the input
        upper_bound: Upper bounds of the input
        maxpool_layer: The MaxPool3d layer to propagate through
        
    Returns:
        Tuple of (lower_bounds, upper_bounds) for the output
    """
    # MaxPool3d is monotonic, so we can directly apply it to bounds
    z_L = F.max_pool3d(
        lower_bound,
        kernel_size=maxpool_layer.kernel_size,
        stride=maxpool_layer.stride,
        padding=maxpool_layer.padding,
        dilation=maxpool_layer.dilation,
        ceil_mode=maxpool_layer.ceil_mode,
        return_indices=False
    )
    
    z_U = F.max_pool3d(
        upper_bound,
        kernel_size=maxpool_layer.kernel_size,
        stride=maxpool_layer.stride,
        padding=maxpool_layer.padding,
        dilation=maxpool_layer.dilation,
        ceil_mode=maxpool_layer.ceil_mode,
        return_indices=False
    )
    
    return z_L, z_U


time: 1.16 s (started: 2026-03-30 12:48:38 +01:00)


In [44]:
from typing import Optional, Tuple
import time

# Interval bound propagation through the CNN3D model
def ibp_propagate_bounds_cnn3d(
    input_tensor: torch.Tensor,
    epsilon: float = 0.1,
    lower_input: Optional[torch.Tensor] = None,
    upper_input: Optional[torch.Tensor] = None,
    skip_first_conv: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Propagate bounds through the entire UCF101SimpleCNN model"""
    # Set initial bounds
    if lower_input is not None and upper_input is not None:
        lower_bounds, upper_bounds = lower_input, upper_input
    else:
        lower_bounds = input_tensor - epsilon
        upper_bounds = input_tensor + epsilon

    # Verify model instance is a UCF101SimpleCNN or a subclass of it
    # assert isinstance(
    #     model, UCF101SimpleCNN
    # ), f"Model is not an instance of UCF101SimpleCNN but {type(model).__name__}"
    assert type(model).__name__ == "UCF101SimpleCNN", f"Model is not a UCF101SimpleCNN but {type(model).__name__}"
    # Process convolutional layers
    # Conv layer 1
    if not skip_first_conv:
        lower_bounds, upper_bounds = _propagate_conv3d(
            lower_bounds, upper_bounds, model.conv1
        )
    
    
    lower_bounds, upper_bounds = _propagate_batchnorm3d(
        lower_bounds, upper_bounds, model.layer1[1]
    )
    lower_bounds, upper_bounds = _propagate_relu(lower_bounds, upper_bounds)
    lower_bounds, upper_bounds = _propagate_maxpool3d(lower_bounds, upper_bounds, model.layer1[3])
    
    # Conv layer 2
    lower_bounds, upper_bounds = _propagate_conv3d(
        lower_bounds, upper_bounds, model.conv2
    )
    lower_bounds, upper_bounds = _propagate_batchnorm3d(
        lower_bounds, upper_bounds, model.layer2[1]
    )
    lower_bounds, upper_bounds = _propagate_relu(lower_bounds, upper_bounds)
    lower_bounds, upper_bounds = _propagate_maxpool3d(lower_bounds, upper_bounds, model.layer2[3])
    
    # Conv layer 3
    lower_bounds, upper_bounds = _propagate_conv3d(
        lower_bounds, upper_bounds, model.conv3
    )
    lower_bounds, upper_bounds = _propagate_batchnorm3d(
        lower_bounds, upper_bounds, model.layer3[1]
    )
    lower_bounds, upper_bounds = _propagate_relu(lower_bounds, upper_bounds)
    lower_bounds, upper_bounds = _propagate_maxpool3d(lower_bounds, upper_bounds, model.layer3[3])
    
    # Global average pooling
    lower_bounds, upper_bounds = _propagate_adaptive_avgpool3d(lower_bounds, upper_bounds, model.global_pool.output_size)
    
    # Flatten
    if lower_bounds.ndim > 2:
        lower_bounds = lower_bounds.view(lower_bounds.size(0), -1)
        upper_bounds = upper_bounds.view(upper_bounds.size(0), -1)
        
    
    # Dropout (during evaluation, it's just identity)
    # We use our custom dropout propagation function
    # lower_bounds, upper_bounds = _propagate_dropout(
    #     lower_bounds, upper_bounds, model.dropout
    # )
    
    # FC layer 1
    lower_bounds, upper_bounds = _propagate_linear(
        lower_bounds, upper_bounds, model.classifier[1]
    )

    return lower_bounds, upper_bounds

time: 69.7 ms (started: 2026-03-30 12:48:39 +01:00)


In [ ]:
start_time = time.time()
ibp_rob_acc = []
epsilons = [0.00001, 0.0001, 0.001, 0.01, 0.1]
for epsilon in epsilons:
    for batch_idx, (x, y) in enumerate(valid_loader):  
        x = x.to(device)
        y = y.to(device)
        verified_correct = 0
        total = x.size(0)

        # print(f"\nIBP Verification with epsilon={epsilon}")
        # print(f"Input shape: {x.shape}")
        # print(f"Number of samples: {total}")

        for i in range(total):
            # print(f"\nVerifying sample {i+1}/{total}")
            input_tensor = x[i:i+1]
            y_i = y[i:i+1]
        
            with torch.no_grad():
                lower_bounds, upper_bounds = ibp_propagate_bounds_cnn3d(
                        input_tensor, epsilon=epsilon
                    )
            
            # Print bound statistics
            # print(f"Lower bounds range: [{lower_bounds.min().item():.4f}, {lower_bounds.max().item():.4f}]")
            # print(f"Upper bounds range: [{upper_bounds.min().item():.4f}, {upper_bounds.max().item():.4f}]")
            # print(f"Bound width: {(upper_bounds - lower_bounds).mean().item():.4f}")

            # Check robustness
            margin = lower_bounds[0, y_i] - torch.max(
                torch.cat([upper_bounds[0, :y_i], upper_bounds[0, y_i + 1 :]], dim=0)
            )

            if margin.item() > 0:
                verified_correct += 1
                # print(f"Sample {i+1} is robust (margin: {margin.item():.4f})")
            # else:
                # print(f"Sample {i+1} is not robust (margin: {margin.item():.4f})")

    ibp_rob_acc.append(verified_correct / total)
print(f"Total time taken: {time.time() - start_time:.2f} seconds")

Total time taken: 240.84 seconds
time: 4min (started: 2026-03-30 12:08:29 +01:00)


In [45]:
# ibp_rob_acc = [0.2636363636363636, 0.23030303030303031, 0.0, 0.0, 0.0]

ibp_rob_acc

[0.4, 0.2, 0.0, 0.0, 0.0]

time: 53.8 ms (started: 2026-03-30 12:48:39 +01:00)


### Selective Interval Bound Propagation Verification

#### Closed form solution Common functions

In [46]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Tuple, List, Dict, Optional, Union
from dataclasses import dataclass
import time
from tqdm import tqdm

@dataclass
class LayerBounds:
    """
    Container for layer activation bounds with utility properties.
    
    Attributes:
        lower: Lower bounds tensor
        upper: Upper bounds tensor
    """
    lower: torch.Tensor
    upper: torch.Tensor
    
    @property
    def center(self) -> torch.Tensor:
        """Compute center of bounds: (L + U) / 2"""
        return (self.lower + self.upper) / 2
    
    @property
    def radius(self) -> torch.Tensor:
        """Compute half-width of bounds: (U - L) / 2"""
        return (self.upper - self.lower) / 2
    
    @property
    def width(self) -> torch.Tensor:
        """Compute total width of bounds: U - L"""
        return self.upper - self.lower
    
    def to(self, device: str) -> 'LayerBounds':
        """Move bounds to specified device."""
        return LayerBounds(
            lower=self.lower.to(device),
            upper=self.upper.to(device)
        )

time: 55 ms (started: 2026-03-30 12:48:39 +01:00)


In [47]:
def closed_form_conv3d_bounds(
    input_tensor: torch.Tensor,
    conv_layer: nn.Conv3d,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    bn_layer: Optional[nn.BatchNorm3d] = None,
    device: str = DEVICE
) -> LayerBounds:
    """
    Compute EXACT bounds on first Conv3D layer output using closed-form solution.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    For Conv3D with (optionally folded) weight W and structured perturbations:
    
    Output at position (o, d', h', w'):
        z[o,d',h',w'] = Σ W[o,i,kd,kh,kw] × x[i, ...] + b[o]
    
    With shared temporal perturbation δ_t = δ_t' for all t, t':
        z = z_clean + Σ W_eff × δ
    
    Where W_eff = Σ_kd W[o,c,kd,h,w] (sum over temporal kernel dim)
    
    For adversarial patches with spatial mask M:
        z = z_clean + Σ W_eff × (M ⊙ δ)
    
    Optimal perturbation:
        δ* = ε × sign(W_eff)
    
    Exact bounds:
        upper = z_clean + ε × Σ|W_eff ⊙ M|
        lower = z_clean - ε × Σ|W_eff ⊙ M|
    
    ============================================================================
    
    Args:
        input_tensor: Input tensor (C, D, H, W) or (B, C, D, H, W)
        conv_layer: First Conv3d layer
        epsilon: Perturbation budget (L∞ norm)
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask (H, W) for patch perturbations
        bn_layer: Optional BatchNorm3d to fold into conv
        device: Computation device
        
    Returns:
        LayerBounds containing exact lower and upper bounds
    """
    # Handle input dimensions
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    B, C, D, H, W = input_tensor.shape
    input_tensor = input_tensor.to(device)
    
    # Get layer parameters (with optional BN folding)
    if bn_layer is not None:
        weights, bias = get_folded_conv_params(conv_layer, bn_layer, device)
    else:
        weights = conv_layer.weight.to(device)
        bias = conv_layer.bias.to(device) if conv_layer.bias is not None else None
    
    out_c, in_c, kd, kh, kw = weights.shape
    
    # Get conv parameters
    stride = conv_layer.stride
    padding = conv_layer.padding
    
    if isinstance(stride, int):
        stride = (stride, stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding, padding)
    
    # Compute clean output
    z_clean = F.conv3d(input_tensor, weights, bias, stride=stride, padding=padding)
    
    # Compute output spatial dimensions
    out_D = (D + 2 * padding[0] - kd) // stride[0] + 1
    out_H = (H + 2 * padding[1] - kh) // stride[1] + 1
    out_W = (W + 2 * padding[2] - kw) // stride[2] + 1
    
    # Create spatial mask if needed
    if spatial_mask is None:
        spatial_mask = torch.ones(H, W, device=device)
    else:
        spatial_mask = spatial_mask.to(device)
    
    # Check if all frames share the same perturbation
    all_shared = (len(shared_frames) == 1 and len(shared_frames[0]) == D)
    
    if all_shared:
        # =================================================================
        # OPTIMIZED PATH: All frames share same perturbation
        # =================================================================
        # Sum weights over temporal kernel dimension
        w_temporal_sum = weights.sum(dim=2)  # (out_c, in_c, kh, kw)
        
        # Create masked perturbation tensor
        # Shape: (B, in_c, H, W)
        spatial_pert = spatial_mask.unsqueeze(0).unsqueeze(0).expand(B, in_c, -1, -1) * epsilon
        
        # Compute radius via 2D convolution with absolute weights
        # Apply spatial stride and padding
        radius_2d = F.conv2d(
            spatial_pert,
            w_temporal_sum.abs(),
            bias=None,
            stride=(stride[1], stride[2]),
            padding=(padding[1], padding[2])
        )  # Shape: (B, out_c, out_H, out_W)
        
        # Expand to match output shape
        radius = radius_2d.unsqueeze(2).expand(-1, -1, out_D, -1, -1)
        
    else:
        # =================================================================
        # GENERAL PATH: Handle arbitrary shared frame configurations
        # =================================================================
        radius = _compute_general_closed_form_radius(
            input_tensor.shape,
            weights,
            epsilon,
            shared_frames,
            spatial_mask,
            conv_layer,
            device
        )
    
    # Compute final bounds
    lower_bounds = z_clean - radius
    upper_bounds = z_clean + radius
    
    return LayerBounds(lower=lower_bounds, upper=upper_bounds)


# =============================================================================
# BatchNorm Folding Utilities
# =============================================================================

def fold_batchnorm_into_conv(
    conv_weight: torch.Tensor,
    conv_bias: Optional[torch.Tensor],
    bn_weight: torch.Tensor,
    bn_bias: torch.Tensor,
    bn_mean: torch.Tensor,
    bn_var: torch.Tensor,
    bn_eps: float = 1e-5
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Fold BatchNorm parameters into Conv weights and bias.
    
    BN(x) = γ × (x - μ) / √(σ² + ε) + β
    
    After folding:
        W_folded = W × γ / √(σ² + ε)
        b_folded = (b - μ) × γ / √(σ² + ε) + β
    
    Args:
        conv_weight: Conv weight tensor (out_c, in_c, kd, kh, kw)
        conv_bias: Conv bias tensor (out_c,) or None
        bn_weight: BatchNorm gamma (out_c,)
        bn_bias: BatchNorm beta (out_c,)
        bn_mean: BatchNorm running mean (out_c,)
        bn_var: BatchNorm running variance (out_c,)
        bn_eps: BatchNorm epsilon for numerical stability
        
    Returns:
        (folded_weight, folded_bias)
    """
    # Compute scale factor: γ / √(σ² + ε)
    std = torch.sqrt(bn_var + bn_eps)
    scale = bn_weight / std
    
    # Fold into weights: reshape scale for broadcasting
    # Weight shape: (out_c, in_c, kd, kh, kw)
    weight_shape = [conv_weight.shape[0]] + [1] * (conv_weight.dim() - 1)
    scale_reshaped = scale.view(*weight_shape)
    
    folded_weight = conv_weight * scale_reshaped
    
    # Fold into bias
    if conv_bias is None:
        conv_bias = torch.zeros(conv_weight.shape[0], device=conv_weight.device)
    
    folded_bias = (conv_bias - bn_mean) * scale + bn_bias
    
    return folded_weight, folded_bias


def get_folded_conv_params(
    conv_layer: nn.Conv3d,
    bn_layer: nn.BatchNorm3d,
    device: str = DEVICE
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Get Conv3d parameters with BatchNorm folded in.
    
    Args:
        conv_layer: Conv3d layer
        bn_layer: BatchNorm3d layer (must be in eval mode)
        device: Computation device
        
    Returns:
        (folded_weight, folded_bias)
    """
    conv_weight = conv_layer.weight.to(device)
    conv_bias = conv_layer.bias.to(device) if conv_layer.bias is not None else None
    
    bn_weight = bn_layer.weight.to(device)
    bn_bias = bn_layer.bias.to(device)
    bn_mean = bn_layer.running_mean.to(device)
    bn_var = bn_layer.running_var.to(device)
    bn_eps = bn_layer.eps
    
    return fold_batchnorm_into_conv(
        conv_weight, conv_bias, bn_weight, bn_bias, bn_mean, bn_var, bn_eps
    )


# =============================================================================
# Closed-Form First Layer Bounds (EXACT)
# =============================================================================

def closed_form_conv3d_bounds(
    input_tensor: torch.Tensor,
    conv_layer: nn.Conv3d,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    bn_layer: Optional[nn.BatchNorm3d] = None,
    device: str = DEVICE
) -> LayerBounds:
    """
    Compute EXACT bounds on first Conv3D layer output using closed-form solution.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    For Conv3D with (optionally folded) weight W and structured perturbations:
    
    Output at position (o, d', h', w'):
        z[o,d',h',w'] = Σ W[o,i,kd,kh,kw] × x[i, ...] + b[o]
    
    With shared temporal perturbation δ_t = δ_t' for all t, t':
        z = z_clean + Σ W_eff × δ
    
    Where W_eff = Σ_kd W[o,c,kd,h,w] (sum over temporal kernel dim)
    
    For adversarial patches with spatial mask M:
        z = z_clean + Σ W_eff × (M ⊙ δ)
    
    Optimal perturbation:
        δ* = ε × sign(W_eff)
    
    Exact bounds:
        upper = z_clean + ε × Σ|W_eff ⊙ M|
        lower = z_clean - ε × Σ|W_eff ⊙ M|
    
    ============================================================================
    
    Args:
        input_tensor: Input tensor (C, D, H, W) or (B, C, D, H, W)
        conv_layer: First Conv3d layer
        epsilon: Perturbation budget (L∞ norm)
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask (H, W) for patch perturbations
        bn_layer: Optional BatchNorm3d to fold into conv
        device: Computation device
        
    Returns:
        LayerBounds containing exact lower and upper bounds
    """
    # Handle input dimensions
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    B, C, D, H, W = input_tensor.shape
    input_tensor = input_tensor.to(device)
    
    # Get layer parameters (with optional BN folding)
    if bn_layer is not None:
        weights, bias = get_folded_conv_params(conv_layer, bn_layer, device)
    else:
        weights = conv_layer.weight.to(device)
        bias = conv_layer.bias.to(device) if conv_layer.bias is not None else None
    
    out_c, in_c, kd, kh, kw = weights.shape
    
    # Get conv parameters
    stride = conv_layer.stride
    padding = conv_layer.padding
    
    if isinstance(stride, int):
        stride = (stride, stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding, padding)
    
    # Compute clean output
    z_clean = F.conv3d(input_tensor, weights, bias, stride=stride, padding=padding)
    
    # Compute output spatial dimensions
    out_D = (D + 2 * padding[0] - kd) // stride[0] + 1
    out_H = (H + 2 * padding[1] - kh) // stride[1] + 1
    out_W = (W + 2 * padding[2] - kw) // stride[2] + 1
    
    # Create spatial mask if needed
    if spatial_mask is None:
        spatial_mask = torch.ones(H, W, device=device)
    else:
        spatial_mask = spatial_mask.to(device)
    
    # Check if all frames share the same perturbation
    all_shared = (len(shared_frames) == 1 and len(shared_frames[0]) == D)
    
    if all_shared:
        # =================================================================
        # OPTIMIZED PATH: All frames share same perturbation
        # =================================================================
        # Sum weights over temporal kernel dimension
        w_temporal_sum = weights.sum(dim=2)  # (out_c, in_c, kh, kw)
        
        # Create masked perturbation tensor
        # Shape: (B, in_c, H, W)
        spatial_pert = spatial_mask.unsqueeze(0).unsqueeze(0).expand(B, in_c, -1, -1) * epsilon
        
        # Compute radius via 2D convolution with absolute weights
        # Apply spatial stride and padding
        radius_2d = F.conv2d(
            spatial_pert,
            w_temporal_sum.abs(),
            bias=None,
            stride=(stride[1], stride[2]),
            padding=(padding[1], padding[2])
        )  # Shape: (B, out_c, out_H, out_W)
        
        # Expand to match output shape
        radius = radius_2d.unsqueeze(2).expand(-1, -1, out_D, -1, -1)
        
    else:
        # =================================================================
        # GENERAL PATH: Handle arbitrary shared frame configurations
        # =================================================================
        radius = _compute_general_closed_form_radius(
            input_tensor.shape,
            weights,
            epsilon,
            shared_frames,
            spatial_mask,
            conv_layer,
            device
        )
    
    # Compute final bounds
    lower_bounds = z_clean - radius
    upper_bounds = z_clean + radius
    
    return LayerBounds(lower=lower_bounds, upper=upper_bounds)


def _compute_general_closed_form_radius(
    input_shape: Tuple[int, ...],
    weights: torch.Tensor,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: torch.Tensor,
    conv_layer: nn.Conv3d,
    device: str
) -> torch.Tensor:
    """
    Compute closed-form radius for general (non-all-shared) frame configurations.
    
    For each shared group g and each output position:
        radius_contribution = ε × |Σ_{frames f ∈ g, kernel k hitting f} W[k] ⊙ M|
    
    Total radius = Σ_g radius_contribution_g
    """
    B, C, D, H, W = input_shape
    out_c, in_c, kd, kh, kw = weights.shape
    
    stride = conv_layer.stride
    padding = conv_layer.padding
    
    if isinstance(stride, int):
        stride = (stride, stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding, padding)
    
    out_D = (D + 2 * padding[0] - kd) // stride[0] + 1
    out_H = (H + 2 * padding[1] - kh) // stride[1] + 1
    out_W = (W + 2 * padding[2] - kw) // stride[2] + 1
    
    # Initialize radius tensor
    radius = torch.zeros(B, out_c, out_D, out_H, out_W, device=device)
    
    # For each shared group, compute its contribution
    for g_idx, group in enumerate(shared_frames):
        group_set = set(group)
        
        # For each output temporal position
        for od in range(out_D):
            # Build mask for which kernel depth positions contribute
            kd_mask = torch.zeros(kd, device=device)
            for kd_idx in range(kd):
                input_d = od * stride[0] + kd_idx - padding[0]
                if 0 <= input_d < D and input_d in group_set:
                    kd_mask[kd_idx] = 1.0
            
            if kd_mask.sum() == 0:
                continue
            
            # Compute effective weight
            w_masked = weights * kd_mask.view(1, 1, kd, 1, 1)
            w_eff = w_masked.sum(dim=2)  # (out_c, in_c, kh, kw)
            
            # Create masked perturbation
            spatial_pert = spatial_mask.unsqueeze(0).unsqueeze(0).expand(B, in_c, -1, -1) * epsilon
            
            # Compute radius contribution via 2D convolution
            radius_2d = F.conv2d(
                spatial_pert,
                w_eff.abs(),
                bias=None,
                stride=(stride[1], stride[2]),
                padding=(padding[1], padding[2])
            )
            
            # Add to radius at this output depth
            radius[:, :, od, :, :] += radius_2d
    
    return radius


time: 59.3 ms (started: 2026-03-30 12:48:39 +01:00)


#### Closed form solution with Lipschitz bounds
In this method we evaluate the closed form solution by computing the effective weight on the first Convolution layer of the network followed by Lipschitz bounds on the remaining layers of the network

In [48]:
# =============================================================================
# Lipschitz-based Bound Propagation (L-2 norm space)
# =============================================================================
def compute_spectral_norm_conv3d(conv_layer: nn.Conv3d) -> float:
    """Compute spectral norm (L2 Lipschitz constant) of Conv3d layer."""
    weight = conv_layer.weight.detach()
    out_c, in_c, kd, kh, kw = weight.shape
    
    weight_2d = weight.reshape(out_c, -1)
    
    try:
        singular_values = torch.linalg.svdvals(weight_2d)
        return singular_values[0].item()
    except Exception:
        return torch.norm(weight_2d, p=2).item()


def compute_spectral_norm_linear(linear_layer: nn.Linear) -> float:
    """Compute spectral norm of Linear layer."""
    weight = linear_layer.weight.detach()
    
    try:
        singular_values = torch.linalg.svdvals(weight)
        return singular_values[0].item()
    except Exception:
        return torch.norm(weight, p=2).item()


def compute_spectral_norm_batchnorm(bn_layer: nn.BatchNorm3d) -> float:
    """
    Compute effective Lipschitz constant of BatchNorm layer.
    
    BN is essentially a diagonal linear transform:
    L = max_i |γ_i / √(σ²_i + ε)|
    """
    gamma = bn_layer.weight.detach()
    var = bn_layer.running_var.detach()
    eps = bn_layer.eps
    
    scale = gamma / torch.sqrt(var + eps)
    return scale.abs().max().item()


def compute_spectral_norm_adaptive_pool(
    input_shape: Tuple[int, int, int],
    output_shape: Tuple[int, int, int]
) -> float:
    """
    Compute Lipschitz constant of AdaptiveAvgPool3d.
    
    For average pooling with kernel size k, the Lipschitz constant is 1.
    (Each output is a weighted average of inputs with weights summing to 1)
    """
    return 1.0




def linf_to_l2_radius(linf_radius: torch.Tensor) -> torch.Tensor:
    """
    Convert L-infinity radius to L-2 radius.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    Given an L-infinity box: x ∈ [center - r, center + r] coordinate-wise,
    we want to find the L-2 radius R such that:
        ||x - center||_2 ≤ R  for all x in the box
    
    The maximum L-2 distance occurs at a corner of the box:
        max_{x in box} ||x - center||_2 = r × √n
    
    where n is the number of dimensions.
    
    However, if the radius varies per dimension (r_i), we have:
        max ||x - center||_2 = √(Σ_i r_i²)
    
    For uniform radius r: R = r × √n
    
    ============================================================================
    SOUNDNESS
    ============================================================================
    
    The L-2 ball with radius R = r × √n contains the entire L-infinity box.
    This is sound but potentially loose (especially for high-dimensional spaces).
    
    Args:
        linf_radius: L-infinity radius tensor (coordinate-wise)
        
    Returns:
        L-2 radius (scalar per sample in batch)
    """
    # Flatten to get all dimensions
    if linf_radius.dim() > 1:
        # Reshape to (batch, -1) to get all spatial/channel dimensions
        batch_size = linf_radius.shape[0]
        linf_flat = linf_radius.view(batch_size, -1)
        # Compute L-2 radius: sqrt(sum of squares of L-inf radii)
        # This is tighter than r * sqrt(n) when radii vary
        l2_radius = torch.norm(linf_flat, p=2, dim=1)  # Shape: (batch_size,)
    else:
        # Single sample, no batch dimension
        linf_flat = linf_radius.view(-1)
        l2_radius = torch.norm(linf_flat, p=2)  # Scalar tensor
    
    return l2_radius


def l2_radius_to_linf_bounds(center: torch.Tensor, l2_radius: torch.Tensor) -> LayerBounds:
    """
    Convert L-2 radius back to L-infinity bounds.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    Given center c and L-2 radius R, we want L-infinity bounds [ℓ, u] such that:
        For all x with ||x - c||_2 ≤ R: ℓ ≤ x ≤ u (coordinate-wise)
    
    The tightest L-infinity box containing the L-2 ball is:
        ℓ_i = c_i - R / √n
        u_i = c_i + R / √n
    
    where n is the number of dimensions.
    
    However, this is conservative. A tighter bound uses the fact that:
        max_{||x-c||_2 ≤ R} |x_i - c_i| = R  (achieved when x = c ± R·e_i)
    
    But since we want coordinate-wise bounds that hold simultaneously:
        ℓ_i = c_i - R
        u_i = c_i + R
    
    This is sound but loose. For tighter bounds, we could use the fact that
    the L-infinity norm of a vector in an L-2 ball is bounded by R.
    
    However, for verification purposes, we use the simpler bound:
        radius_linf = R / √n  (per dimension)
    
    This ensures that the L-infinity box contains the L-2 ball.
    
    Args:
        center: Center tensor (can have batch dimension)
        l2_radius: L-2 radius (scalar or tensor with batch dimension)
        
    Returns:
        LayerBounds with L-infinity bounds
    """
    # Ensure l2_radius is a tensor
    if not isinstance(l2_radius, torch.Tensor):
        l2_radius = torch.tensor(l2_radius, dtype=center.dtype, device=center.device)
    else:
        l2_radius = l2_radius.to(center.device)
    
    # Get number of dimensions (excluding batch)
    has_batch = center.dim() > 1
    if has_batch:
        batch_size = center.shape[0]
        n_dims = center.numel() // batch_size
        
        # Ensure l2_radius has batch dimension
        if l2_radius.dim() == 0:
            l2_radius = l2_radius.unsqueeze(0)
        if l2_radius.shape[0] == 1 and batch_size > 1:
            l2_radius = l2_radius.expand(batch_size)
        elif l2_radius.shape[0] != batch_size:
            raise ValueError(f"l2_radius batch size {l2_radius.shape[0]} doesn't match center batch size {batch_size}")
    else:
        # No batch dimension - ensure l2_radius is scalar
        n_dims = center.numel()
        if l2_radius.dim() > 0:
            if l2_radius.numel() > 1:
                raise ValueError("l2_radius has multiple values but center has no batch dimension")
            l2_radius = l2_radius.squeeze()
    
    # Convert to L-infinity radius per dimension
    # Using the bound: ||x||_∞ ≤ ||x||_2 / √n
    sqrt_n_dims = torch.sqrt(torch.tensor(n_dims, dtype=center.dtype, device=center.device))
    linf_radius_per_dim = l2_radius / sqrt_n_dims
    
    # Expand to match center shape
    if has_batch:
        linf_radius = linf_radius_per_dim.view(batch_size, *([1] * (center.dim() - 1)))
    else:
        linf_radius = linf_radius_per_dim
    
    lower = center - linf_radius
    upper = center + linf_radius
    
    return LayerBounds(lower=lower, upper=upper)


def l2_to_linf_radius(l2_radius: torch.Tensor, n_dims: int, device: str) -> torch.Tensor:
    """Convert L-2 radius to L-infinity radius per dimension."""
    sqrt_n = torch.sqrt(torch.tensor(n_dims, dtype=torch.float32, device=device))
    return l2_radius / sqrt_n



def lipschitz_propagate_cnn3d(
    first_layer_bounds: LayerBounds,
    model,
    device: str = DEVICE
) -> Tuple[LayerBounds, float]:
    """
    Propagate bounds through CNN3D using Lipschitz constants.
    
    Converts L∞ bounds to L2 radius, propagates via Lipschitz, converts back.
    
    Args:
        first_layer_bounds: Bounds from first layer
        model: CNN3D model
        device: Computation device
        
    Returns:
        (final_bounds, cumulative_lipschitz_constant)
    """
    # Ensure batch dimension
    lower = first_layer_bounds.lower
    upper = first_layer_bounds.upper
    
    if lower.dim() == 4:
        lower = lower.unsqueeze(0)
        upper = upper.unsqueeze(0)
    
    lower = lower.to(device)
    upper = upper.to(device)
    
    # Center-radius representation
    center = (lower + upper) / 2
    radius_linf = (upper - lower) / 2
    
    # Convert to L2 radius
    l2_radius = linf_to_l2_radius(radius_linf)
    
    cumulative_lipschitz = 1.0
    
    # BN1 Lipschitz constant (layer1[1] is the BatchNorm3d after conv1)
    lip_bn1 = compute_spectral_norm_batchnorm(model.layer1[1])
    l2_radius = l2_radius * lip_bn1
    cumulative_lipschitz *= lip_bn1

    # ReLU after layer1 (1-Lipschitz)
    n_dims = center.numel() // center.shape[0]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)

    lower_tmp = center - linf_r.view(-1, 1, 1, 1, 1)
    upper_tmp = center + linf_r.view(-1, 1, 1, 1, 1)
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius_linf = (upper_out - lower_out) / 2
    l2_radius = linf_to_l2_radius(radius_linf)

    # MaxPool1 (1-Lipschitz): apply to center, radius unchanged
    center = model.layer1[3](center)

    # Conv2 + BN2 (layer2[1] is the BatchNorm3d after conv2)
    w2_folded, b2_folded = get_folded_conv_params(model.conv2, model.layer2[1], device)
    center = F.conv3d(center, w2_folded, b2_folded,
                      stride=model.conv2.stride, padding=model.conv2.padding)

    lip_conv2 = compute_spectral_norm_conv3d(model.conv2)
    lip_bn2 = compute_spectral_norm_batchnorm(model.layer2[1])
    l2_radius = l2_radius * lip_conv2 * lip_bn2
    cumulative_lipschitz *= lip_conv2 * lip_bn2

    # ReLU after layer2 (1-Lipschitz)
    n_dims = center.numel() // center.shape[0]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)

    lower_tmp = center - linf_r.view(-1, 1, 1, 1, 1)
    upper_tmp = center + linf_r.view(-1, 1, 1, 1, 1)
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius_linf = (upper_out - lower_out) / 2
    l2_radius = linf_to_l2_radius(radius_linf)

    # MaxPool2 (1-Lipschitz): apply to center, radius unchanged
    center = model.layer2[3](center)

    # Conv3 + BN3 (layer3[1] is the BatchNorm3d after conv3)
    w3_folded, b3_folded = get_folded_conv_params(model.conv3, model.layer3[1], device)
    center = F.conv3d(center, w3_folded, b3_folded,
                      stride=model.conv3.stride, padding=model.conv3.padding)

    lip_conv3 = compute_spectral_norm_conv3d(model.conv3)
    lip_bn3 = compute_spectral_norm_batchnorm(model.layer3[1])
    l2_radius = l2_radius * lip_conv3 * lip_bn3
    cumulative_lipschitz *= lip_conv3 * lip_bn3

    # ReLU after layer3 (1-Lipschitz)
    n_dims = center.numel() // center.shape[0]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)

    lower_tmp = center - linf_r.view(-1, 1, 1, 1, 1)
    upper_tmp = center + linf_r.view(-1, 1, 1, 1, 1)
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius_linf = (upper_out - lower_out) / 2
    l2_radius = linf_to_l2_radius(radius_linf)

    # MaxPool3 (1-Lipschitz): apply to center, radius unchanged
    center = model.layer3[3](center)

    # Global Average Pooling (1-Lipschitz)
    center = model.global_pool(center)

    # Flatten
    center = center.view(center.size(0), -1)

    # Classifier FC (single linear layer: classifier[1])
    center = F.linear(center, model.classifier[1].weight, model.classifier[1].bias)
    lip_fc = compute_spectral_norm_linear(model.classifier[1])
    l2_radius = l2_radius * lip_fc
    cumulative_lipschitz *= lip_fc

    # Convert back to L∞ bounds
    n_dims = center.shape[1]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)

    final_lower = center - linf_r.unsqueeze(1)
    final_upper = center + linf_r.unsqueeze(1)

    return LayerBounds(lower=final_lower, upper=final_upper), cumulative_lipschitz


time: 59 ms (started: 2026-03-30 12:48:39 +01:00)


In [49]:
def closed_form_lipschitz_verify_cnn3d(
    model: CNN3D,
    input_tensor: torch.Tensor,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    device: str = DEVICE
) -> Tuple[torch.Tensor, torch.Tensor, float, float]:
    """
    Complete verification: Closed-form first layer + Lipschitz propagation.
    
    Args:
        model: CNN3D model
        input_tensor: Input tensor
        epsilon: Perturbation budget
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask
        device: Computation device
        
    Returns:
        (lower_bounds, upper_bounds, runtime_seconds, lipschitz_constant)
    """
    start_time = time.time()
    
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    model = model.to(device)
    model.eval()
    input_tensor = input_tensor.to(device)
    
    # Step 1: Closed-form bounds on first Conv3D + BN1
    first_layer_bounds = closed_form_conv3d_bounds(
        input_tensor,
        model.conv1,
        epsilon,
        shared_frames,
        spatial_mask,
        model.layer1[1],
        device
    )
    
    # Step 2: Lipschitz propagation
    final_bounds, lipschitz_const = lipschitz_propagate_cnn3d(
        first_layer_bounds,
        model,
        device
    )
    
    runtime = time.time() - start_time
    
    return final_bounds.lower, final_bounds.upper, runtime, lipschitz_const


time: 52.3 ms (started: 2026-03-30 12:48:39 +01:00)


In [ ]:
from models.models import *
from utils.train_validate_3dcnn import *
from typing import *

model_path = SAVED_MODELS_PATH / "best_simple3dcnn_ucf101.pth"
# model = CNN3D(t_dim=len(selected_frames), img_x=img_x, img_y=img_y,
#               drop_p=dropout, fc_hidden1=fc_hidden1,  fc_hidden2=fc_hidden2, 
#               num_classes=num_classes, in_channels=num_channels,
#               ).to(device)

model = UCF101SimpleCNN(num_frames=num_frames, img_x=img_x, img_y=img_y,
              num_classes=num_classes, in_channels=num_channels, dropout_rate=dropout,
              ).to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

In [50]:
closed_form_lips_rob_acc = []
clean_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.to(DEVICE).to(torch.long)
                        
        final_lower, final_upper, _, _ = closed_form_lipschitz_verify_cnn3d(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_lips_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-Lipschitz Robust Acc: {closed_form_lips_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-Lipschitz Robust Accuracy: ", closed_form_lips_rob_acc)

# print(correct," ",robust, " ",total)

Epsilon: 0.1 => Clean Acc: 0.00%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 0.01 => Clean Acc: 0.00%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 0.001 => Clean Acc: 0.00%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 0.0001 => Clean Acc: 0.00%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 1e-05 => Clean Acc: 0.00%, Closed-form-Lipschitz Robust Acc: 58.44%
Model Accuracy:  [0.0, 0.0, 0.0, 0.0, 0.0]
Closed-form-Lipschitz Robust Accuracy:  [0.0, 0.0, 0.0, 0.0, 58.43843843843843]
time: 42.6 s (started: 2026-03-30 12:48:39 +01:00)


In [ ]:
closed_form_lips_rob_acc = []
clean_acc = []

epsilons = [3, 2.75, 2.50, 2.25, 2, 1.75, 1.5, 1.25, 1, 0.75, 0.5, 0.25, 0.1, 0.075, 0.05, 0.025, 0.01, 0.0075, 0.005, 0.0025, 0.001, 0.0001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4,5,6,7,8,9]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.to(DEVICE).to(torch.long)

        final_lower, final_upper, _, _ = closed_form_lipschitz_verify_cnn3d(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_lips_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-Lipschitz Robust Acc: {closed_form_lips_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-Lipschitz Robust Accuracy: ", closed_form_lips_rob_acc)

# print(correct," ",robust, " ",total)

In [13]:
gurobi_params = {
    "Method": 1,  # Dual simplex
    "Presolve": 2,  # Aggressive presolve
    "NumericFocus": 3,  # Highest numerical accuracy
    "TimeLimit": 60,  # 60 second timeout
    "OutputFlag": 0,  # Suppress output
    "Threads": 1,  # Single thread per model
    "FeasibilityTol": 1e-6,  # Tighter feasibility tolerance
    "OptimalityTol": 1e-6,  # Tighter optimality tolerance
}

In [14]:
import gurobipy as gp
import numpy as np
from gurobipy import GRB
from copy import deepcopy
from typing import Dict, List, Optional, Tuple



def generate_perturbation_constraints(
    input_tensor: torch.Tensor,
    frames: List[List[int]],
    epsilons: List[float],
    kernel_size: int,
    stride: int = 1,
    padding: int = 0,
) -> Tuple[torch.Tensor, Dict, Dict, Dict]:
    """
    Generates constraints on the perturbation tensor based on input frames and epsilon bounds.

    Args:
        input_tensor: The base single input tensor (C, D, H, W)
        frames: Each sublist specifies a group of frames with shared perturbation constraints
        epsilons: Perturbation bounds for each frame group in `frames`
        kernel_size: Kernel size of the convolutional layer
        stride: Stride of the convolutional layer
        padding: Padding applied to the convolutional layer

    Returns:
        Tuple containing:
        - perturbation_tensor: The tensor where perturbations will be applied
        - bounds_dict: Bounds on the perturbation values
        - equality_dict: Fixed values for frames not in specified groups
        - shared_dict: Dictionary for shared perturbation constraints
    """
    C, D, H, W = input_tensor.shape  # Channels, Depth, Height, Width

    # Initialize the perturbation tensor to zeros
    perturbation_tensor = torch.zeros_like(input_tensor)

    bounds_dict = {}
    equality_dict = {}
    shared_dict = {}

    # Maximum valid index in the flattened tensor to avoid index overflow
    max_index = C * D * H * W

    # Iterate over each frame group to set constraints within receptive fields
    for group_index, frame_group in enumerate(frames):
        epsilon = epsilons[group_index]

        # Iterate over each channel
        for c in range(C):
            # Use the first frame in the group as the reference for bounds
            first_frame = frame_group[0]
            for h in range(H):  # Iterate over height of frame
                for w in range(W):  # Iterate over width of frame
                    # Calculate the flat index for the first frame in the group
                    first_frame_index = (
                        (c * D * H * W) + (first_frame * H * W) + (h * W) + w
                    )

                    # Ensure first_frame_index is within bounds
                    if first_frame_index < max_index:
                        # Set bounds for the first frame in the group
                        bounds_dict[first_frame_index] = {
                            "lower": -epsilon,
                            "upper": epsilon,
                        }

                    # For other frames in the group, set as shared with the first frame
                    for f in frame_group[1:]:  # Skip the first frame
                        # Calculate the flat index for other frames in the group
                        flat_index = (c * D * H * W) + (f * H * W) + (h * W) + w

                        # Check if indices are within bounds to prevent index errors
                        if flat_index < max_index and first_frame_index < max_index:
                            shared_dict[flat_index] = first_frame_index

    # Populate equality_dict with strict values for frames not in specified groups
    frames_set = set(
        f for group in frames for f in group
    )  # All frames in specified groups

    for c in range(C):  # Iterate over each channel
        for d in range(D):  # Iterate over each depth (frame)
            if d not in frames_set:  # Frames not in specified groups
                for h in range(H):  # Iterate over each height position
                    for w in range(W):  # Iterate over each width position
                        # Calculate flat index for the current position
                        flat_index = (c * D * H * W) + (d * H * W) + (h * W) + w
                        # Ensure the index is valid and not in other dictionaries
                        if (
                            flat_index < max_index
                            and flat_index not in bounds_dict
                            and flat_index not in shared_dict
                        ):
                            # Set equality constraints in equality_dict, keeping perturbation strictly zero
                            equality_dict[flat_index] = 0.0

    return perturbation_tensor, bounds_dict, equality_dict, shared_dict

def compute_3D_conv_bounds(
    input_tensor: torch.Tensor,
    perturbation_tensor: torch.Tensor,
    bounds_dict: Dict,
    equality_dict: Dict,
    shared_dict: Dict,
    kernel_size: int,
    stride: int = 1,
    padding: int = 0,
    conv_layer: nn.Module = None,
) -> List[List[Tuple[float, float]]]:
    """
    Efficiently computes the bounds on the output of the first 3D convolutional layer given perturbation constraints.

    Args:
        input_tensor: The base input tensor
        perturbation_tensor: The perturbation tensor with applied constraints
        bounds_dict: Bounds on the perturbation values
        equality_dict: Fixed values for frames not in specified groups
        shared_dict: Dictionary for shared perturbation constraints
        kernel_size: Kernel size of the convolutional layer
        stride: Stride of the convolutional layer
        padding: Padding applied to the convolutional layer
        conv_layer: The first convolutional layer

    Returns:
        List of computed lower and upper bounds for each output element of the first convolutional layer
    """
    # Extract first convolutional layer weights and biases
    weights = conv_layer.weight.cpu().detach().numpy()
    biases = conv_layer.bias.cpu().detach().numpy()
    out_channels, in_channels, kernel_d, kernel_h, kernel_w = weights.shape

    # Flatten input and perturbation tensors
    base_input = input_tensor.flatten().cpu().detach().numpy()
    C, D, H, W = input_tensor.shape

    # Identify which input indices have perturbations (i.e., not fixed)
    perturbed_indices = set(bounds_dict.keys())

    # Initialize Gurobi model with parameters
    m = gp.Model("efficient_conv_perturb")
    m.setParam("OutputFlag", 0)  # Suppress Gurobi output
    # for param, value in gurobi_params.items():
    #     m.setParam(param, value)

    # Create perturbation variables only for perturbed inputs
    perturb_vars = {}
    for idx in perturbed_indices:
        lb = bounds_dict[idx].get("lower", -GRB.INFINITY)
        ub = bounds_dict[idx].get("upper", GRB.INFINITY)
        perturb_vars[idx] = m.addVar(lb=lb, ub=ub, name=f"perturb_{idx}")

    # Apply shared constraints: tie perturbation variables if necessary
    for idx1, idx2 in shared_dict.items():
        if idx1 in perturb_vars and idx2 in perturb_vars:
            m.addConstr(
                perturb_vars[idx1] == perturb_vars[idx2],
                name=f"shared_{idx1}_{idx2}",
            )

    m.update()

    # Prepare output bounds storage
    output_bounds = []

    # Compute output dimensions after convolution
    D_out = (D + 2 * padding - kernel_d) // stride + 1
    H_out = (H + 2 * padding - kernel_h) // stride + 1
    W_out = (W + 2 * padding - kernel_w) // stride + 1

    # Loop over output channels
    for out_channel in range(out_channels):
        channel_bounds = []

        # Loop over each output spatial location
        for d_out in range(D_out):
            for h_out in range(H_out):
                for w_out in range(W_out):
                    const_term = 0.0
                    perturb_expr = gp.LinExpr()

                    # For each receptive field contributing to this output
                    for in_channel in range(in_channels):
                        for kd in range(kernel_d):
                            for kh in range(kernel_h):
                                for kw in range(kernel_w):
                                    input_d = d_out * stride + kd - padding
                                    input_h = h_out * stride + kh - padding
                                    input_w = w_out * stride + kw - padding

                                    if (
                                        0 <= input_d < D
                                        and 0 <= input_h < H
                                        and 0 <= input_w < W
                                    ):
                                        flat_index = (
                                            in_channel * D * H * W
                                            + input_d * H * W
                                            + input_h * W
                                            + input_w
                                        )
                                        weight = weights[
                                            out_channel, in_channel, kd, kh, kw
                                        ]

                                        if flat_index in perturb_vars:
                                            perturb_expr += (
                                                weight * perturb_vars[flat_index]
                                            )
                                        else:
                                            # Fixed input, contribute directly
                                            const_term += (
                                                weight * base_input[flat_index]
                                            )

                    # Add bias
                    const_term += biases[out_channel]

                    # Solve for lower bound
                    m.setObjective(perturb_expr + const_term, GRB.MINIMIZE)
                    try:
                        m.optimize()
                        if m.status == GRB.OPTIMAL:
                            lower_bound = m.ObjVal
                        else:
                            print(f"Warning: Lower bound optimization failed with status {m.status}")
                            # Use IBP bounds as fallback
                            lower_bound = const_term - abs(perturb_expr.getCoeff().sum() * max(abs(bounds_dict[idx]["lower"]) for idx in perturb_vars))
                    except Exception as e:
                        print(f"Error computing lower bound: {str(e)}")
                        lower_bound = float("-inf")

                    # Solve for upper bound
                    m.setObjective(perturb_expr + const_term, GRB.MAXIMIZE)
                    try:
                        m.optimize()
                        if m.status == GRB.OPTIMAL:
                            upper_bound = m.ObjVal
                        else:
                            print(f"Warning: Upper bound optimization failed with status {m.status}")
                            # Use IBP bounds as fallback
                            upper_bound = const_term + abs(perturb_expr.getCoeff().sum() * max(abs(bounds_dict[idx]["upper"]) for idx in perturb_vars))
                    except Exception as e:
                        print(f"Error computing upper bound: {str(e)}")
                        upper_bound = float("inf")

                    # Sanity check
                    if upper_bound < lower_bound:
                        print(f"Warning: Upper bound ({upper_bound}) < Lower bound ({lower_bound})")
                        # Use IBP bounds as fallback
                        lower_bound = const_term - abs(perturb_expr.getCoeff().sum() * max(abs(bounds_dict[idx]["lower"]) for idx in perturb_vars))
                        upper_bound = const_term + abs(perturb_expr.getCoeff().sum() * max(abs(bounds_dict[idx]["upper"]) for idx in perturb_vars))

                    channel_bounds.append((lower_bound, upper_bound))

        output_bounds.append(channel_bounds)

    return output_bounds

def format_bounds_for_next_layer(
        input_tensor: torch.Tensor, output_bounds: List[List[Tuple[float, float]]], conv_layer: nn.Module
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Formats the computed bounds from a convolutional layer to be used as input bounds for the next layer.

    Args:
        input_tensor: The input batch tensor with shape (batch_size, channels, depth, height, width)
        output_bounds: Bounds computed by `compute_3D_conv_bounds`

    Returns:
        Tuple of (lower_bounds, upper_bounds) tensors formatted for the next layer input
    """
    # Get first input sample in the batch and compute output shape after applying the first conv layer
    single_input = input_tensor  # Shape (C, D, H, W)
    conv_output = conv_layer(single_input)
    single_output_shape = conv_output.shape  # Exclude batch dimension (C, D, H, W)

    out_channels = single_output_shape[0]
    output_shape = (
        single_output_shape  # Output shape of the conv layer for individual inputs
    )

    # Initialize tensors for lower and upper bounds with computed shape
    lower_bounds = torch.empty(output_shape)
    upper_bounds = torch.empty(output_shape)

    # Flatten the output bounds list for easy indexing
    bounds_flat = [
        bound for channel_bounds in output_bounds for bound in channel_bounds
    ]

    # Reshape bounds into tensors of the correct shape for the next layer
    index = 0
    for c in range(out_channels):
        for d in range(output_shape[1]):  # Depth dimension
            for h in range(output_shape[2]):  # Height dimension
                for w in range(output_shape[3]):  # Width dimension
                    lower, upper = bounds_flat[index]
                    if lower is not None:
                        lower_bounds[c, d, h, w] = lower
                    else:
                        print("err")
                        lower_bounds[c, d, h, w] = float(
                            "-inf"
                        )  # We can replace with IBP bounds later if we want
                    if upper is not None:
                        upper_bounds[c, d, h, w] = upper
                    else:
                        upper_bounds[c, d, h, w] = float("inf")
                    index += 1

    lower_bounds = lower_bounds.to(device)
    upper_bounds = upper_bounds.to(device)
    return lower_bounds, upper_bounds

In [15]:
def selective_interval_bound_propagation_cnn3d(
        input_tensor: torch.Tensor,
        epsilons: List[float],
        sets_of_frames: List[List[int]],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Selective Interval Bound propagation with LP applied to the first layer of the network

        Args:
            input_tensor: Input tensor
            epsilons: List of epsilon values for perturbation
            sets_of_frames: List of lists each corresponding to one set of shared perturbed frames

        Returns:
            Tuple of (lower_bounds, upper_bounds) for the entire batch
        """
        lower_bounds, upper_bounds = None, None

        perturbation_tensor, bounds_dict, equality_dict, shared_dict = (
            generate_perturbation_constraints(
                input_tensor[0],
                sets_of_frames,
                epsilons,
                kernel_size=2,
                stride=1,
                padding=0,
            )
        )

        for i in range(len(input_tensor)):
            # Compute bounds for the first layer using compute_3D_conv_bounds
            output_bounds = compute_3D_conv_bounds(
                input_tensor[i],
                perturbation_tensor,
                bounds_dict,
                equality_dict,
                shared_dict,
                kernel_size=2,
                stride=1,
                padding=0,
                conv_layer=model.conv1,
            )

            # Format bounds for the next layer
            lower_bounds_first, upper_bounds_first = format_bounds_for_next_layer(
                input_tensor[i], output_bounds, model.conv1
            )
            
            
            lower_bounds_first = lower_bounds_first.unsqueeze(0)
            upper_bounds_first = upper_bounds_first.unsqueeze(0)


            lb, ub = ibp_propagate_bounds_cnn3d(
                input_tensor[i].unsqueeze(0),
                lower_input=lower_bounds_first,
                upper_input=upper_bounds_first,
                skip_first_conv=True,
            )

            if lower_bounds is None and upper_bounds is None:
                lower_bounds = lb
                upper_bounds = ub
            else:
                lower_bounds = torch.cat((lower_bounds, lb), dim=0)
                upper_bounds = torch.cat((upper_bounds, ub), dim=0)

        return lower_bounds, upper_bounds

### SKIP SIBP full verification for now

In [ ]:
start_time = time.time()
sibp_rob_acc = []
epsilons = [0.00001, 0.0001, 0.001, 0.01, 0.1]
for epsilon in epsilons:
    for batch_idx, (x, y) in enumerate(valid_loader):  
        x = x.to(device)
        y = y.to(device)
        verified_correct = 0
        total = x.size(0)

        # print(f"\nIBP Verification with epsilon={epsilon}")
        # print(f"Input shape: {x.shape}")
        # print(f"Number of samples: {total}")

        for i in range(total):
            # print(f"\nVerifying sample {i+1}/{total}")
            input_tensor = x[i:i+1]
            y_i = y[i:i+1]
        
            with torch.no_grad():
                lower_bounds, upper_bounds = selective_interval_bound_propagation_cnn3d(
                        input_tensor, epsilons=[epsilon], sets_of_frames= [list(range(1, 29))]
                    )
            
            # Print bound statistics
            # print(f"Lower bounds range: [{lower_bounds.min().item():.4f}, {lower_bounds.max().item():.4f}]")
            # print(f"Upper bounds range: [{upper_bounds.min().item():.4f}, {upper_bounds.max().item():.4f}]")
            # print(f"Bound width: {(upper_bounds - lower_bounds).mean().item():.4f}")

            # Check robustness
            margin = lower_bounds[0, y_i] - torch.max(
                torch.cat([upper_bounds[0, :y_i], upper_bounds[0, y_i + 1 :]], dim=0)
            )

            if margin.item() > 0:
                verified_correct += 1
            #     print(f"Sample {i+1} is robust (margin: {margin.item():.4f})")
            # else:
            #     print(f"Sample {i+1} is not robust (margin: {margin.item():.4f})")

    sibp_rob_acc.append(verified_correct / total)
print(f"Total time taken: {time.time() - start_time:.2f} seconds")


### S-IBP Patch Verification

In [16]:
# import random

# Set random seed for reproducibility
# random.seed(42)

# Select 100 random samples
# random_test_list = random.sample(test_list, k=100)
# random_test_label = [test_label[test_list.index(x)] for x in random_test_list]

# Reload the validation dataset with image names
valid_set = Dataset_3DCNN_MidFrames(DATA_PATH, test_list, test_label, selected_frames, transform=transform, is_image_names_needed=True, num_frames=num_frames, is_RGB=True)

# load UCF101 actions names
params = {'batch_size': 2, 'shuffle': False, 'num_workers': 4, 'pin_memory': True} if use_cuda else {}
                       
valid_loader = data.DataLoader(valid_set, **params)

In [17]:
def generate_perturbation_constraints_patch(
        input_tensor: torch.Tensor,
        frame_patch_locations: Dict[int, List[Tuple[int, int, int, int]]],
        epsilon: float,
    ) -> Tuple[torch.Tensor, Dict, Dict, Dict]:
        """
        Generate perturbation constraints for a 3D CNN under patch constraints.

        Args:
            input_tensor: The base input tensor (C, D, H, W)
            frame_patch_locations: Dictionary mapping frame indices to lists of (x, y, w, h) tuples
            epsilon: Maximum allowed perturbation

        Returns:
            Tuple containing:
            - perturbation_tensor: The tensor where perturbations will be applied
            - bounds_dict: Bounds on the perturbation values
            - equality_dict: Fixed values for frames not in specified groups
            - shared_dict: Dictionary for shared perturbation constraints
        """
        C, D, H, W = input_tensor.shape
        bounds_dict = {}
        shared_dict = {}
        patch_shared_indices = {}

        # Iterate through each frame and its list of bounding boxes
        for frame_idx, boxes in frame_patch_locations.items():
            for x, y, w, h in boxes:
                for c in range(C):
                    for i in range(w):
                        for j in range(h):
                            h_idx = x + i
                            w_idx = y + j
                            if 0 <= frame_idx < D and 0 <= h_idx < H and 0 <= w_idx < W:
                                flat_index = (
                                    c * D * H * W
                                    + frame_idx * H * W
                                    + h_idx * W
                                    + w_idx
                                )
                                bounds_dict[flat_index] = {
                                    "lower": -epsilon,
                                    "upper": epsilon,
                                }

                                key = (c, i, j)
                                if key in patch_shared_indices:
                                    shared_dict[flat_index] = patch_shared_indices[key]
                                else:
                                    patch_shared_indices[key] = flat_index

        perturbation_tensor = torch.zeros_like(input_tensor)
        equality_dict = {}  # Not needed for patch version

        return perturbation_tensor, bounds_dict, equality_dict, shared_dict
    

def compute_3D_conv_bounds_patch(
    input_tensor: torch.Tensor,
    perturbation_tensor: torch.Tensor,
    bounds_dict: Dict,
    equality_dict: Dict,
    shared_dict: Dict,
    kernel_size: int,
    stride: int = 1,
    padding: int = 0,
    conv_layer: nn.Module = None,
) -> List[List[Tuple[float, float]]]:
    """
    Computes the bounds on the output of the first 3D convolutional layer given perturbation constraints.

    Args:
        input_tensor: The base input tensor
        perturbation_tensor: The perturbation tensor (not used)
        bounds_dict: Bounds on the perturbation values
        equality_dict: Not used (for compatibility)
        shared_dict: Dictionary for shared perturbation constraints
        kernel_size: Kernel size of the convolutional layer
        stride: Stride of the convolutional layer
        padding: Padding applied to the convolutional layer
        conv_layer: The first convolutional layer to compute bounds for

    Returns:
        List of computed lower and upper bounds for each output element of the first convolutional layer
    """

    weights = conv_layer.weight.cpu().detach().numpy()
    biases = conv_layer.bias.cpu().detach().numpy()
    out_channels, in_channels, kernel_d, kernel_h, kernel_w = weights.shape

    base_input = input_tensor.flatten().cpu().detach().numpy()
    C, D, H, W = input_tensor.shape

    perturbed_indices = set(bounds_dict.keys())

    m = gp.Model("conv_bounds_patch")
    m.setParam("OutputFlag", 0)

    perturb_vars = {}
    for idx in perturbed_indices:
        lb = bounds_dict[idx].get("lower", -GRB.INFINITY)
        ub = bounds_dict[idx].get("upper", GRB.INFINITY)
        perturb_vars[idx] = m.addVar(lb=lb, ub=ub, name=f"perturb_{idx}")

    for idx1, idx2 in shared_dict.items():
        if idx1 in perturb_vars and idx2 in perturb_vars:
            m.addConstr(
                perturb_vars[idx1] == perturb_vars[idx2],
                name=f"shared_{idx1}_{idx2}",
            )

    m.update()

    output_bounds = []

    D_out = (D + 2 * padding - kernel_d) // stride + 1
    H_out = (H + 2 * padding - kernel_h) // stride + 1
    W_out = (W + 2 * padding - kernel_w) // stride + 1

    for out_channel in range(out_channels):
        channel_bounds = []

        for d_out in range(D_out):
            for h_out in range(H_out):
                for w_out in range(W_out):
                    const_term = 0.0
                    perturb_expr = gp.LinExpr()

                    for in_channel in range(in_channels):
                        for kd in range(kernel_d):
                            for kh in range(kernel_h):
                                for kw in range(kernel_w):
                                    input_d = d_out * stride + kd - padding
                                    input_h = h_out * stride + kh - padding
                                    input_w = w_out * stride + kw - padding

                                    if (
                                        0 <= input_d < D
                                        and 0 <= input_h < H
                                        and 0 <= input_w < W
                                    ):
                                        flat_index = (
                                            in_channel * D * H * W
                                            + input_d * H * W
                                            + input_h * W
                                            + input_w
                                        )
                                        weight = weights[
                                            out_channel, in_channel, kd, kh, kw
                                        ]

                                        if flat_index in perturb_vars:
                                            perturb_expr += (
                                                weight * perturb_vars[flat_index]
                                            )
                                        else:
                                            const_term += (
                                                weight * base_input[flat_index]
                                            )

                    const_term += biases[out_channel]

                    if perturb_expr.size() == 0:
                        lower_bound = upper_bound = const_term
                    else:
                        m.setObjective(perturb_expr + const_term, GRB.MINIMIZE)
                        m.optimize()
                        lower_bound = (
                            m.ObjVal if m.status == GRB.OPTIMAL else const_term
                        )

                        m.setObjective(perturb_expr + const_term, GRB.MAXIMIZE)
                        m.optimize()
                        upper_bound = (
                            m.ObjVal if m.status == GRB.OPTIMAL else const_term
                        )

                    channel_bounds.append((float(lower_bound), float(upper_bound)))

        output_bounds.append(channel_bounds)

    return output_bounds

def format_bounds_for_next_layer(
        input_tensor: torch.Tensor, output_bounds: List[List[Tuple[float, float]]], conv_layer: nn.Module
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Formats the computed bounds from a convolutional layer to be used as input bounds for the next layer.

    Args:
        input_tensor: The input batch tensor with shape (batch_size, channels, depth, height, width)
        output_bounds: Bounds computed by `compute_3D_conv_bounds`

    Returns:
        Tuple of (lower_bounds, upper_bounds) tensors formatted for the next layer input
    """
    # Get first input sample in the batch and compute output shape after applying the first conv layer
    single_input = input_tensor  # Shape (C, D, H, W)
    conv_output = conv_layer(single_input)
    single_output_shape = conv_output.shape  # Exclude batch dimension (C, D, H, W)

    out_channels = single_output_shape[0]
    output_shape = (
        single_output_shape  # Output shape of the conv layer for individual inputs
    )

    # Initialize tensors for lower and upper bounds with computed shape
    lower_bounds = torch.empty(output_shape)
    upper_bounds = torch.empty(output_shape)

    # Flatten the output bounds list for easy indexing
    bounds_flat = [
        bound for channel_bounds in output_bounds for bound in channel_bounds
    ]

    # Reshape bounds into tensors of the correct shape for the next layer
    index = 0
    for c in range(out_channels):
        for d in range(output_shape[1]):  # Depth dimension
            for h in range(output_shape[2]):  # Height dimension
                for w in range(output_shape[3]):  # Width dimension
                    lower, upper = bounds_flat[index]
                    if lower is not None:
                        lower_bounds[c, d, h, w] = lower
                    else:
                        print("err")
                        lower_bounds[c, d, h, w] = float(
                            "-inf"
                        )  # We can replace with IBP bounds later if we want
                    if upper is not None:
                        upper_bounds[c, d, h, w] = upper
                    else:
                        upper_bounds[c, d, h, w] = float("inf")
                    index += 1

    lower_bounds = lower_bounds.to(device)
    upper_bounds = upper_bounds.to(device)
    return lower_bounds, upper_bounds

In [18]:
def process_bounding_boxes(
     bounding_box_info: Dict, H: int, W: int
) -> Dict[int, List[Tuple[int, int, int, int]]]:
    """
    Process bounding box information into frame_patch_locations format.

    Args:
        bounding_box_info: Dictionary containing bounding box information for each object
        H: Height of the video frames
        W: Width of the video frames

    Returns:
        Dictionary with frame indices as keys and lists of (x1, y1, x2, y2) tuples as values
    """
    frame_patch_locations = {}

    # Process each object's bounding boxes
    for obj_id, obj_data in bounding_box_info.items():
        for frame_data in obj_data["bounding_box"]:
            # Extract frame number from image name (e.g., "frame000001.jpg" -> 1)
            frame_num = int(frame_data["image"].split("frame")[1].split(".")[0]) - 1

            # Get normalized bounding box coordinates
            x1, y1, x2, y2 = frame_data["xyxyn"]

            # Convert normalized coordinates to pixel coordinates
            x1_px = int(x1 * W)
            y1_px = int(y1 * H)
            x2_px = int(x2 * W)
            y2_px = int(y2 * H)

            # Store the coordinates in frame_patch_locations as a list
            if frame_num not in frame_patch_locations:
                frame_patch_locations[frame_num] = []
            frame_patch_locations[frame_num].append((x1_px, y1_px, x2_px, y2_px))

    return frame_patch_locations




In [19]:
def selective_interval_bound_propagation_patch_cnn3d(
        input_tensor: torch.Tensor, bounding_box_info: Dict, epsilon: float
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Selective Interval Bound propagation with LP applied to the first layer of the network for patch-based verification

        Args:
            input_tensor: Input tensor
            bounding_box_info: Dictionary containing patch information
            epsilon: Epsilon perturbation value

        Returns:
            Tuple of (lower_bounds, upper_bounds) for the entire batch
        """
        lower_bounds, upper_bounds = None, None

        frame_patch_locations = process_bounding_boxes(
            bounding_box_info, input_tensor.shape[3], input_tensor.shape[4]
        )

        for i in range(len(input_tensor)):
            # Generate perturbation constraints
            perturbation_tensor, bounds_dict, equality_dict, shared_dict = (
                generate_perturbation_constraints_patch(
                    input_tensor[i], frame_patch_locations, epsilon
                )
            )

            # Get bounds for the first layer using MILP
            output_bounds = compute_3D_conv_bounds_patch(
                input_tensor[i],
                perturbation_tensor,
                bounds_dict,
                equality_dict,
                shared_dict,
                kernel_size=2,
                stride=1,
                padding=0,
                conv_layer=model.conv1,
            )

            # Format bounds for the next layer
            lower_bounds_first, upper_bounds_first = format_bounds_for_next_layer(
                input_tensor[i], output_bounds, model.conv1
            )
            

            lower_bounds_first = lower_bounds_first.unsqueeze(0)
            upper_bounds_first = upper_bounds_first.unsqueeze(0)
            
            # Perform IBP for the remaining layers
            lb, ub = ibp_propagate_bounds_cnn3d(
                input_tensor[i].unsqueeze(0),
                lower_input=lower_bounds_first,
                upper_input=upper_bounds_first,
                skip_first_conv=True,
            )

            if lower_bounds is None and upper_bounds is None:
                lower_bounds = lb
                upper_bounds = ub
            else:
                lower_bounds = torch.cat((lower_bounds, lb), dim=0)
                upper_bounds = torch.cat((upper_bounds, ub), dim=0)

        return lower_bounds, upper_bounds

In [20]:
import json
def get_bounding_box_info(bounding_box_path: str, folder_name: str, json_file_name: str) -> Dict:
    """
    Load bounding box information from a JSON file.

    Args:
        bounding_box_path: Path to the JSON file containing bounding box information

    Returns:
        Dictionary containing bounding box information
    """
    with open(os.path.join(bounding_box_path,folder_name,json_file_name), "r") as f:
        bounding_box_info = json.load(f)
    return bounding_box_info

In [ ]:
start_time = time.time()
sibp_patch_rob_acc = []
epsilons = [0.00001, 0.0001, 0.001, 0.01, 0.1]
for epsilon in epsilons:
    for batch_idx, (x, y, image_ids) in enumerate(valid_loader):  
        x = x.to(device)
        y = y.to(device)
        verified_correct = 0
        total = x.size(0)

        # print(f"\nIBP Verification with epsilon={epsilon}")
        # print(f"Input shape: {x.shape}")
        # print(f"Number of samples: {total}")

        for i in range(total):
            # print(f"\nVerifying sample {i+1}/{total}")
            input_tensor = x[i:i+1]
            y_i = y[i:i+1]
        
            with torch.no_grad():
                bounding_box_info = get_bounding_box_info(
                    BOUNDING_BOX_PATH, f"{valid_loader.dataset.folders[batch_idx*batch_size + i]}_tracking", BOUNDING_BOX_JSON_FILE_NAME
                )
                # print(bounding_box_info)
                lower_bounds, upper_bounds = selective_interval_bound_propagation_patch_cnn3d(
                        input_tensor, bounding_box_info, epsilon=epsilon
                    )
                
            
            # Print bound statistics
            # print(f"Lower bounds range: [{lower_bounds.min().item():.4f}, {lower_bounds.max().item():.4f}]")
            # print(f"Upper bounds range: [{upper_bounds.min().item():.4f}, {upper_bounds.max().item():.4f}]")
            # print(f"Bound width: {(upper_bounds - lower_bounds).mean().item():.4f}")

            # Check robustness
            margin = lower_bounds[0, y_i] - torch.max(
                torch.cat([upper_bounds[0, :y_i], upper_bounds[0, y_i + 1 :]], dim=0)
            )

            if margin.item() > 0:
                verified_correct += 1
                # print(f"Sample {i+1} is robust (margin: {margin.item():.4f})")
            # else:
            #     print(f"Sample {i+1} is not robust (margin: {margin.item():.4f})")
            break
    sibp_patch_rob_acc.append(verified_correct / total)
    break
print(f"Total time taken: {time.time() - start_time:.2f} seconds")

Set parameter Username
Set parameter LicenseID to value 2620395
Academic license - for non-commercial use only - expires 2026-02-10


In [ ]:
sibp_patch_rob_acc